# Cartographie prédictive GWS — ElasticNet

Notebook Google Colab généré à partir du script Python d'origine : **`Elastiknet.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Chaque cellule exécute directement son étape, comme le script d'origine (qui n'a pas de fonction `main()`).
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_ELASTICNET` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS - ELASTICNET UNIQUEMENT
#  (TRAIN = longue période, TEST = 5% des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  - Modèle ElasticNet (sklearn) sur grille GLDAS (~25 km)
#
#  - Variables explicatives (sections dans name_files.txt) :
#       * Precip (CHIRPS_Clipped OU Precipitations_IDW OU Precipitations_API_IDW)
#       * MODIS_LST_Clipped     → LST (journalier)
#       * MODIS_NDVI_Clipped    → NDVI (16 jours, étendu à toutes les dates)
#       * MODIS_ET_Clipped      → ET (~8 jours, étendu à toutes les dates)
#       * ERA5_Qsb_Clipped      → Qsb (journalier)
#       * SMAP_RZSM_Clipped     → RZSM (journalier)
#       * SMAP_SoilM_Clipped    → SoilM (journalier)
#
#  - Cible :
#       * GLDAS_GWS_Clipped     → GWS (journalier)
#
#  - Features finales :
#       [PRECIP, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, sinDOY, cosDOY]
#
#  - Rééchantillonnage en respectant correctement les NoData
#  - Remplissage des NaN d'entrée → cartes SANS TROUS
#    à l’intérieur des Regions (asset GEE "Regions")
#  - Clip de toutes les cartes (test + forecast) sur la zone
#    Regions reconstruite à partir des rasters GEE
#
#  ➤ Périodes :
#    POOL TRAIN : 2021-01-01 → 2025-06-30
#    TEST = 5% des dates tirées aléatoirement dans cette fenêtre
#    FORECAST : 2025-07-01 → 2025-10-30
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab (scikit-learn, numpy et joblib le sont).

In [ ]:
!pip install -q rasterio

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
import joblib

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 1. PARAMÈTRES GÉNÉRAUX

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1) PARAMÈTRES GÉNÉRAUX
# ------------------------------------------------------------
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_ELASTICNET")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE = -9999.0
TEST_FRACTION = 0.05

# Priorité des sections precipitation (on prend la 1ère trouvée)
PREFERRED_PRECIP_SECTIONS = [
    "CHIRPS_Clipped",
    "Precipitations_API_IDW",
    "Precipitations_IDW",
    "CHIRPS",
]

## 2. FENÊTRES TEMPORELLES

In [ ]:
# ------------------------------------------------------------
# 2) FENÊTRES TEMPORELLES
# ------------------------------------------------------------
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = datetime(2025, 7, 1)
FORECAST_END   = datetime(2025, 10, 30)

print("=== PARAMÈTRES GÉNÉRAUX ===")
print(" BASE_DIR :", BASE_DIR)
print(" NAME_FILE :", NAME_FILE)
print(" OUT_DIR :", OUT_DIR)
print(" TRAIN_START/END :", TRAIN_START, "→", TRAIN_END)
print(" TEST_FRACTION :", TEST_FRACTION)
print(" FORECAST_START/END :", FORECAST_START.date(), "→", FORECAST_END.date())
print("============================\n")

### 2.1 DATES EXCLUES (LST manquante 2022-10-11 → 2022-10-22)

In [ ]:
# ------------------------------------------------------------
# 2.1) DATES EXCLUES (LST manquante 2022-10-11 → 2022-10-22)
# ------------------------------------------------------------
EXCLUDED_DATES = set(date(2022, 10, day) for day in range(11, 23))

print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)

## 3. FONCTIONS UTILITAIRES

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 3) FONCTIONS UTILITAIRES
# ------------------------------------------------------------
def extract_date_from_filename(path):
    """Extrait YYYYMMDD depuis le nom de fichier."""
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire la date depuis : {name}")
    return datetime.strptime(m.group(1), "%Y%m%d").date()

def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path):
    """Parse name_files.txt -> dict {SectionName: [relative/path.tif, ...]}"""
    sections = {}
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections

def _norm_section(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.lower())

def resolve_section_name(sections: dict, wanted: str, aliases=None) -> str:
    """Trouve une section même si le nom varie (casse, underscores, etc.)."""
    if aliases is None:
        aliases = []
    norm_map = {_norm_section(k): k for k in sections.keys()}

    for candidate in [wanted] + list(aliases):
        nk = _norm_section(candidate)
        if nk in norm_map:
            return norm_map[nk]

    wanted_n = _norm_section(wanted)
    for k in sections.keys():
        kn = _norm_section(k)
        if wanted_n in kn or kn in wanted_n:
            return k

    available = "\n - " + "\n - ".join(sorted(sections.keys()))
    raise RuntimeError(
        f"Section '{wanted}' introuvable. Sections disponibles :{available}"
    )

def resolve_precip_section(sections: dict) -> str:
    """Choisit automatiquement la bonne section precipitation."""
    for candidate in PREFERRED_PRECIP_SECTIONS:
        try:
            return resolve_section_name(sections, candidate)
        except RuntimeError:
            pass
    # Si aucune trouvée :
    available = "\n - " + "\n - ".join(sorted(sections.keys()))
    raise RuntimeError(
        "Aucune section precipitation trouvée. J'ai cherché : "
        f"{PREFERRED_PRECIP_SECTIONS}\nSections disponibles :{available}"
    )

def build_date_to_path_dict_from_section(sections, section_name):
    """Construit dict {date: path_abs} pour une section donnée."""
    if section_name not in sections:
        available = "\n - " + "\n - ".join(sorted(sections.keys()))
        raise RuntimeError(
            f"Section '{section_name}' absente de {NAME_FILE}. Sections :{available}"
        )

    d = {}
    for rel_path in sections[section_name]:
        abs_path = os.path.normpath(os.path.join(BASE_DIR, rel_path))
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            date_key = extract_date_from_filename(abs_path)
            d[date_key] = abs_path
        except ValueError as e:
            print("⚠️", e)
    return d

def read_raster_raw(path):
    """Lit un raster bande 1 float32, sans modifier NoData."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata

def to_float_with_nan(arr, nodata):
    """Convertit en float32 et remplace nodata -> NaN."""
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

def reproject_to_template(
    src_arr,
    src_profile,
    template_profile,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average,
):
    """Rééchantillonne src_arr sur la grille template_profile."""
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32",
    )

    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata

    reproject(**kwargs)
    return dst_arr

def fill_nan_with_mean(arr):
    """Remplace les NaN par la moyenne (si possible)."""
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)

def build_prev_date_support(date_dict):
    """Fonction get_prev(date) -> dernière date <= date."""
    dates_sorted = sorted(date_dict.keys())

    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]

    return get_prev

# Masque Regions reconstruit
REGION_MASK = None

def update_region_mask(precip_res, lst_res, ndvi_res, template_profile):
    global REGION_MASK
    base_mask = ~(np.isnan(precip_res) & np.isnan(lst_res) & np.isnan(ndvi_res))
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit à partir des rasters (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask

def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé, on prend toute la grille.")
        REGION_MASK = np.ones(
            (template_profile["height"], template_profile["width"]),
            dtype=bool,
        )
    return REGION_MASK

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## 4. INDEXATION DES FICHIERS PAR DATE (via name_files.txt)

In [ ]:
# ------------------------------------------------------------
# 4) INDEXATION DES FICHIERS PAR DATE (via name_files.txt)
# ------------------------------------------------------------
print("\n=== INDEXATION DES FICHIERS PAR DATE (via name_files.txt) ===")
sections = parse_name_file(NAME_FILE)

print("\n📌 Sections détectées dans name_files.txt :")
for k in sorted(sections.keys()):
    print(" -", k)

# Résolution robuste des sections
SEC_PRECIP = resolve_precip_section(sections)
SEC_LST    = resolve_section_name(sections, "MODIS_LST_Clipped", aliases=["LST", "MODIS_LST"])
SEC_NDVI   = resolve_section_name(sections, "MODIS_NDVI_Clipped", aliases=["NDVI", "MODIS_NDVI"])
SEC_ET     = resolve_section_name(sections, "MODIS_ET_Clipped", aliases=["ET", "MODIS_ET"])
SEC_GWS    = resolve_section_name(sections, "GLDAS_GWS_Clipped", aliases=["GWS", "GLDAS_GWS"])
SEC_QSB    = resolve_section_name(sections, "ERA5_Qsb_Clipped", aliases=["Qsb", "ERA5_Qsb", "ERA5_QSB"])
SEC_RZSM   = resolve_section_name(sections, "SMAP_RZSM_Clipped", aliases=["RZSM", "SMAP_RZSM"])
SEC_SOILM  = resolve_section_name(sections, "SMAP_SoilM_Clipped", aliases=["SoilM", "SMAP_SoilM", "SMAP_SOILM"])

print("\n✅ Sections utilisées :")
print(" PRECIP :", SEC_PRECIP)
print(" LST    :", SEC_LST)
print(" NDVI   :", SEC_NDVI)
print(" ET     :", SEC_ET)
print(" GWS    :", SEC_GWS)
print(" QSB    :", SEC_QSB)
print(" RZSM   :", SEC_RZSM)
print(" SOILM  :", SEC_SOILM)

precip_dict = build_date_to_path_dict_from_section(sections, SEC_PRECIP)
lst_dict    = build_date_to_path_dict_from_section(sections, SEC_LST)
ndvi_dict   = build_date_to_path_dict_from_section(sections, SEC_NDVI)
et_dict     = build_date_to_path_dict_from_section(sections, SEC_ET)
gws_dict    = build_date_to_path_dict_from_section(sections, SEC_GWS)
qsb_dict    = build_date_to_path_dict_from_section(sections, SEC_QSB)
rzsm_dict   = build_date_to_path_dict_from_section(sections, SEC_RZSM)
soilm_dict  = build_date_to_path_dict_from_section(sections, SEC_SOILM)

print("\nNb dates PRECIP :", len(precip_dict))
print("Nb dates LST    :", len(lst_dict))
print("Nb dates NDVI   :", len(ndvi_dict))
print("Nb dates ET     :", len(et_dict))
print("Nb dates GWS    :", len(gws_dict))
print("Nb dates QSB    :", len(qsb_dict))
print("Nb dates RZSM   :", len(rzsm_dict))
print("Nb dates SOILM  :", len(soilm_dict))

# Support NDVI/ET (non journaliers)
get_ndvi_date_for = build_prev_date_support(ndvi_dict)
get_et_date_for   = build_prev_date_support(et_dict)

## 5. DATES TRAIN / TEST (5% des dates)

In [ ]:
# ------------------------------------------------------------
# 5) DATES TRAIN / TEST (5% des dates)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DES DATES TRAIN / TEST (5% des dates pour TEST) ===")

all_common_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(gws_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

print("Nombre total de dates communes (PRECIP ∩ LST ∩ GWS ∩ QSB ∩ RZSM ∩ SOILM) :",
      len(all_common_dates))
if not all_common_dates:
    raise RuntimeError("⚠️ Aucune date commune entre les variables journalières.")

date_candidates = [
    d for d in all_common_dates
    if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
]
if len(date_candidates) == 0:
    raise RuntimeError("⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusion).")

train_dates, test_dates = train_test_split(
    date_candidates,
    test_size=TEST_FRACTION,
    random_state=42,
    shuffle=True,
)

train_dates = sorted(train_dates)
test_dates  = sorted(test_dates)

print(f"\nFenêtre TRAIN (pool) : {TRAIN_START} → {TRAIN_END}")
print(f" → Nombre de dates train : {len(train_dates)}")
print(f" → TEST = {TEST_FRACTION*100:.1f}% : {len(test_dates)} dates")

## 6. CONSTRUCTION DATASET (TRAIN + TEST)

In [ ]:
# ------------------------------------------------------------
# 6) CONSTRUCTION DATASET (TRAIN + TEST)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DU DATASET (toutes les dates TRAIN + TEST) ===")

X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []
template_profile = None

def build_Xy_for_date(date_key, template_profile, role="train"):
    # NDVI étendu
    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(f"⏭️ [{role}] {date_key} : aucun NDVI disponible (<= {date_key}), ignoré.")
        return None, None, template_profile

    # ET étendu
    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(f"⏭️ [{role}] {date_key} : aucun ET disponible (<= {date_key}), ignoré.")
        return None, None, template_profile

    # GWS (template)
    if date_key not in gws_dict:
        print(f"⏭️ [{role}] {date_key} : GWS manquant, ignoré.")
        return None, None, template_profile

    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    if template_profile is None:
        template_profile = gws_prof
        print("\nGrille GLDAS (template) :")
        print(" - CRS  :", template_profile["crs"])
        print(" - Taille :", template_profile["width"], "x", template_profile["height"])
        print(" - Résolution approx (deg) :", template_profile["transform"][0])

    # Variables journalières
    if (
        date_key not in precip_dict
        or date_key not in lst_dict
        or date_key not in qsb_dict
        or date_key not in rzsm_dict
        or date_key not in soilm_dict
    ):
        print(f"⏭️ [{role}] {date_key} : au moins une variable journalière manquante, ignoré.")
        return None, None, template_profile

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile,
                                       src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw, lst_prof, template_profile,
                                       src_nodata=lst_nodata, dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw, ndvi_prof, template_profile,
                                       src_nodata=ndvi_nodata, dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw, et_prof, template_profile,
                                       src_nodata=et_nodata, dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw, qsb_prof, template_profile,
                                       src_nodata=qsb_nodata, dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw, rzsm_prof, template_profile,
                                       src_nodata=rzsm_nodata, dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw, soilm_prof, template_profile,
                                       src_nodata=soilm_nodata, dst_nodata=np.nan)

    # Update Regions mask
    update_region_mask(precip_res, lst_res, ndvi_res, template_profile)
    region_mask = get_region_mask(template_profile)

    # Fill NaN (no holes)
    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    mask_valid = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_valid):
        print(f"⏭️ [{role}] {date_key} : aucun pixel valide, ignoré.")
        return None, None, template_profile

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels = np.count_nonzero(mask_valid)
    sin_plane = np.full(n_pixels, sin_doy, dtype="float32")
    cos_plane = np.full(n_pixels, cos_doy, dtype="float32")

    X_day = np.stack(
        [
            precip_f[mask_valid],
            lst_f[mask_valid],
            ndvi_f[mask_valid],
            et_f[mask_valid],
            qsb_f[mask_valid],
            rzsm_f[mask_valid],
            soilm_f[mask_valid],
            sin_plane,
            cos_plane,
        ],
        axis=1,
    )
    y_day = gws_arr[mask_valid]

    row_mask = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
    if not np.any(row_mask):
        print(f"⏭️ [{role}] {date_key} : pixels invalides (NaN), ignoré.")
        return None, None, template_profile

    X_day = X_day[row_mask]
    y_day = y_day[row_mask]

    print(f" ✔️ [{role}] {date_key} : {X_day.shape[0]} pixels utilisés.")
    return X_day, y_day, template_profile

# Boucle train
for d in train_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="train")
    if X_day is None:
        continue
    X_train_list.append(X_day)
    y_train_list.append(y_day)

# Boucle test
for d in test_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="test")
    if X_day is None:
        continue
    X_test_list.append(X_day)
    y_test_list.append(y_day)

if len(X_train_list) == 0:
    raise RuntimeError("⚠️ Aucun pixel d'entraînement valide n'a été trouvé.")
if len(X_test_list) == 0:
    raise RuntimeError("⚠️ Aucun pixel de test valide n'a été trouvé.")

X_train = np.vstack(X_train_list).astype("float32")
y_train = np.concatenate(y_train_list).astype("float32")
X_test  = np.vstack(X_test_list).astype("float32")
y_test  = np.concatenate(y_test_list).astype("float32")

print(f"\nNombre total de pixels d'entraînement : {X_train.shape[0]}")
print(f"Nombre total de pixels de test        : {X_test.shape[0]}")
print("Exemple feature train :", X_train[0])

## 7. MODÈLE ELASTICNET

In [ ]:
# ------------------------------------------------------------
# 7) MODÈLE ELASTICNET
# ------------------------------------------------------------
print("\n=== ENTRAÎNEMENT DU MODÈLE ELASTICNET ===")

enet = ElasticNet(
    alpha=0.001,     # à ajuster
    l1_ratio=0.5,    # 0=L2, 1=L1, 0.5=mix
    max_iter=10000,
    random_state=42,
)

t0 = time.time()
enet.fit(X_train, y_train)
t1 = time.time()
training_time_ms = (t1 - t0) * 1000.0

# Train metrics
y_train_pred = enet.predict(X_train)
r2_train   = r2_score(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train  = mean_absolute_error(y_train, y_train_pred)

# Test metrics
y_test_pred = enet.predict(X_test)
r2_test   = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test  = mean_absolute_error(y_test, y_test_pred)

print("\n╔═════════ MODÈLE ELASTICNET ═════════╗")
print(f" ➤ Temps d'entraînement : {training_time_ms:.2f} ms")
print(f" ➤ R² TRAIN : {r2_train:.4f}")
print(f" ➤ MAE TRAIN : {mae_train:.4f}")
print(f" ➤ RMSE TRAIN: {rmse_train:.4f}")
print("---------------------------------------------")
print(f" ➤ R² TEST : {r2_test:.4f}")
print(f" ➤ MAE TEST : {mae_test:.4f}")
print(f" ➤ RMSE TEST: {rmse_test:.4f}")
print("╚══════════════════════════════════════╝")

# Sauvegarde du modèle
model_path = os.path.join(OUT_DIR, "ElasticNet_GWS_model.joblib")
joblib.dump(enet, model_path)
print(f"\n💾 Modèle ElasticNet sauvegardé : {model_path}")

## 8. ÉVALUATION CARTOGRAPHIQUE SUR DATES TEST

In [ ]:
# ------------------------------------------------------------
# 8) ÉVALUATION CARTOGRAPHIQUE SUR DATES TEST
# ------------------------------------------------------------
print("\n=== ÉVALUATION CARTOGRAPHIQUE SUR TOUTES LES DATES TEST ===")
region_mask = get_region_mask(template_profile)

all_true, all_pred = [], []

out_profile_test = template_profile.copy()
out_profile_test.update(dtype="float32", count=1, nodata=NODATA_VALUE)

for date_key in test_dates:
    print(f"\n🧪 Test cartographique : {date_key}")

    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(" ⏭️ NDVI/ET indisponible -> ignoré.")
        continue

    # GWS vérité terrain
    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    # Vérification journalières
    if (
        date_key not in precip_dict
        or date_key not in lst_dict
        or date_key not in qsb_dict
        or date_key not in rzsm_dict
        or date_key not in soilm_dict
    ):
        print(" ⏭️ Variable journalière manquante -> ignoré.")
        continue

    # Lecture
    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    # Reproject
    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw, et_prof, template_profile, src_nodata=et_nodata, dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw, qsb_prof, template_profile, src_nodata=qsb_nodata, dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw, soilm_prof, template_profile, src_nodata=soilm_nodata, dst_nodata=np.nan)

    # Fill NaN
    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    # DOY
    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2*np.pi*doy/365.0)
    cos_doy = np.cos(2*np.pi*doy/365.0)
    n_pixels_full = precip_f.size
    sin_plane = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane = np.full(n_pixels_full, cos_doy, dtype="float32")

    # X_all
    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane,
            cos_plane,
        ],
        axis=1,
    )

    if np.isnan(X_all).any():
        print(" ⏭️ X_all contient des NaN -> ignoré.")
        continue

    y_pred_all = enet.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    mask_gws = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_gws):
        print(" ⚠️ Aucun pixel valide GWS pour cette date -> métriques ignorées.")
        continue

    y_true = gws_arr[mask_gws]
    y_pred = pred_clip[mask_gws]

    r2_d = r2_score(y_true, y_pred)
    mae_d = mean_absolute_error(y_true, y_pred)
    rmse_d = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f" ➤ R²  : {r2_d:.4f}")
    print(f" ➤ MAE : {mae_d:.4f}")
    print(f" ➤ RMSE: {rmse_d:.4f}")

    all_true.append(y_true)
    all_pred.append(y_pred)

    out_name = os.path.join(OUT_DIR, f"GWS_TEST_PRED_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile_test) as dst:
        dst.write(pred_clip.astype("float32"), 1)
    print(f" ✔️ Carte test sauvegardée : {out_name}")

if len(all_true) > 0:
    all_true_flat = np.concatenate(all_true)
    all_pred_flat = np.concatenate(all_pred)
    r2_global = r2_score(all_true_flat, all_pred_flat)
    mae_global = mean_absolute_error(all_true_flat, all_pred_flat)
    rmse_global = np.sqrt(mean_squared_error(all_true_flat, all_pred_flat))
    print("\n📊 MÉTRIQUES GLOBALES TEST")
    print(f" ➤ R² global : {r2_global:.4f}")
    print(f" ➤ MAE global : {mae_global:.4f}")
    print(f" ➤ RMSE global: {rmse_global:.4f}")
else:
    print("\n⚠️ Pas de métriques globales (aucun pixel valide).")

## 9. PRÉDICTION CARTOGRAPHIQUE (FORECAST)

In [ ]:
# ------------------------------------------------------------
# 9) PRÉDICTION CARTOGRAPHIQUE (FORECAST)
# ------------------------------------------------------------
print("\n=== PHASE DE PRÉDICTION CARTOGRAPHIQUE (FORECAST) ===")

out_profile = template_profile.copy()
out_profile.update(dtype="float32", count=1, nodata=NODATA_VALUE)

all_forecast_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

if len(all_forecast_dates) == 0:
    print("⚠️ Aucune date commune pour forecast -> arrêt.")
else:
    print("Plage possible forecast (journalières) :", min(all_forecast_dates), "→", max(all_forecast_dates))

user_start = FORECAST_START.date()
user_end   = FORECAST_END.date()

forecast_start_date = max(user_start, min(all_forecast_dates)) if all_forecast_dates else user_start
forecast_end_date   = min(user_end,   max(all_forecast_dates)) if all_forecast_dates else user_end

forecast_dates = [
    d for d in all_forecast_dates
    if (forecast_start_date <= d <= forecast_end_date) and (d not in EXCLUDED_DATES)
]

print("Fenêtre demandée :", user_start, "→", user_end)
print("Fenêtre utilisée :", forecast_start_date, "→", forecast_end_date)
print("Nb dates forecast :", len(forecast_dates))

region_mask = get_region_mask(template_profile)

for date_key in forecast_dates:
    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(f"⏭️ {date_key} : NDVI/ET indisponible -> ignoré.")
        continue

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw, et_prof, template_profile, src_nodata=et_nodata, dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw, qsb_prof, template_profile, src_nodata=qsb_nodata, dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw, soilm_prof, template_profile, src_nodata=soilm_nodata, dst_nodata=np.nan)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2*np.pi*doy/365.0)
    cos_doy = np.cos(2*np.pi*doy/365.0)

    n_pixels_full = precip_f.size
    sin_plane = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane = np.full(n_pixels_full, cos_doy, dtype="float32")

    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane,
            cos_plane,
        ],
        axis=1,
    )

    if np.isnan(X_all).any():
        print(f"⏭️ {date_key} : X_all contient des NaN -> ignoré.")
        continue

    y_pred_all = enet.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    out_name = os.path.join(OUT_DIR, f"GWS_PRED_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile) as dst:
        dst.write(pred_clip.astype("float32"), 1)

    print(f" ✔️ Forecast sauvegardé : {out_name}")

# Récapitulatif
tif_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".tif")]
print(f"\n📂 Total cartes générées dans {OUT_DIR} : {len(tif_files)}")

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)